In [1]:
import torch
import numpy as np

In [ ]:
# input: (*, H_in), weight: (H_out, H_in), bias: (H_out)

class BitLinear(torch.nn.Module):

    def __init__(self, in_features, out_features, bias=True):

        super().__init__()

        self.in_features = in_features
        self.out_features = out_features

        self.weight = torch.nn.Parameter(torch.Tensor(out_features, in_features, dtype = torch.bool))

        if bias:
            self.bias = torch.nn.Parameter(torch.Tensor(out_features), dtype = torch.bool)

        else:
            self.register_parameter('bias', None)
            
        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.uniform_(self.bias, 0, 1).round()
        if self.bias is not None:
            
            torch.nn.init.uniform_(self.bias, 0, 1).round()

    def forward(self, input):

        output = torch.empty_like((*input.shape[:-1], self.out_features), dtype = torch.bool)
        
        for i in range(self.in_features):
            for j in range(self.out_features):
                output[..., i, j] = torch.logical_xor(torch.logical_or(torch.logical_and(input[..., i], self.weight.T[..., j])), self.bias[j])

        return output

In [22]:
a = np.array([[1, 0, 1], [0, 1, 0]]).astype(bool)
b = np.array([[1, 0], [0, 1], [1, 0]]).astype(bool)
c = np.dot(a, b)
print(c)

[[ True False]
 [False  True]]


In [24]:
a = torch.tensor(np.array([[1, 0, 1], [0, 1, 0]]))
b = torch.tensor(np.array([[1, 0], [0, 1], [1, 0]]))
c = torch.nn.functional.linear(a, b.T)
print(c)

tensor([[2, 0],
        [0, 1]])


In [ ]:
a = torch.tensor(np.array([[1, 0, 1], [0, 1, 0]]).astype(bool), dtype = torch.bool)
b = torch.tensor(np.array([[1, 0], [0, 1], [1, 0]]).astype(bool), dtype = torch.bool)

RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1